# Fabric Cost Intelligence

Real $ cost per workspace per day. Reads from Delta tables created by the Data Prep notebook.

**Zero API calls** - all data comes from the Lakehouse Delta tables.

**Prerequisite:** Run `Fabric Monitoring Data Prep` first.

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

df_billing = spark.sql("SELECT * FROM billing_fabric").toPandas()
dim_capacities = spark.sql("SELECT * FROM capacity_capacities").toPandas()
dim_workspaces = spark.sql("SELECT * FROM capacity_workspaces").toPandas()
dim_domains = spark.sql("SELECT * FROM chargeback_domains").toPandas()
df_chargeback = spark.sql("SELECT * FROM chargeback_facts").toPandas()

df_billing["date"] = pd.to_datetime(df_billing["date"])
df_billing["costInBillingCurrency"] = pd.to_numeric(df_billing["costInBillingCurrency"], errors="coerce").fillna(0)
df_chargeback["Date"] = pd.to_datetime(df_chargeback["Date"])
df_chargeback = df_chargeback.merge(dim_domains, on="Domain unique key", how="left")

currency = str(df_billing["billingCurrency"].dropna().iloc[0]) if not df_billing.empty else "USD"

print(f"Billing:    {len(df_billing):,} rows | {df_billing['date'].min().date()} to {df_billing['date'].max().date()}")
print(f"Chargeback: {len(df_chargeback):,} rows | {df_chargeback['Date'].min().date()} to {df_chargeback['Date'].max().date()}")
print(f"Capacities: {len(dim_capacities)} | Workspaces: {len(dim_workspaces)} | Domains: {len(dim_domains)}")
print(f"Currency: {currency}")

---
## 1. Daily Cost per Capacity

In [ ]:
if not df_billing.empty:
    cost_daily_cap = (
        df_billing.groupby(["capacity_name", "date"])
        .agg(daily_cost=pd.NamedAgg(column="costInBillingCurrency", aggfunc="sum"),
             meter_count=pd.NamedAgg(column="meterName", aggfunc="nunique"))
        .reset_index()
    )
    cost_daily_cap = cost_daily_cap.merge(
        dim_capacities[["Capacity Id", "Capacity Name", "Capacity Name Lower", "SKU", "Core count"]],
        left_on="capacity_name", right_on="Capacity Name Lower", how="left"
    )
    unmatched = cost_daily_cap[cost_daily_cap["Capacity Id"].isna()]["capacity_name"].unique()
    if len(unmatched) > 0:
        print(f"WARNING: {len(unmatched)} unmatched: {list(unmatched)}")
        cost_daily_cap = cost_daily_cap[cost_daily_cap["Capacity Id"].notna()].copy()
    print(f"Daily cost: {len(cost_daily_cap):,} rows | Capacities: {cost_daily_cap['Capacity Name'].nunique()}")
    for _, r in cost_daily_cap.groupby(["Capacity Name", "SKU"]).agg(total=pd.NamedAgg(column="daily_cost", aggfunc="sum"), avg=pd.NamedAgg(column="daily_cost", aggfunc="mean"), days=pd.NamedAgg(column="date", aggfunc="nunique")).reset_index().sort_values("total", ascending=False).iterrows():
        print(f"  {r['Capacity Name']} ({r['SKU']}): {currency} {r['total']:,.2f} total | {currency} {r['avg']:.2f}/day ({r['days']}d)")
else:
    cost_daily_cap = pd.DataFrame()
    print("No billing data. Run Data Prep first.")

---
## 2. Allocate Cost to Workspaces

workspace_cost = capacity_daily_cost x (workspace_CU / total_capacity_CU)

In [ ]:
if not cost_daily_cap.empty:
    ws_daily_cu = df_chargeback.groupby(["Capacity Id", "Workspace Id", "Date"]).agg(ws_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum")).reset_index()
    cap_daily_cu = ws_daily_cu.groupby(["Capacity Id", "Date"]).agg(cap_total_cu=pd.NamedAgg(column="ws_cu", aggfunc="sum")).reset_index()
    ws_daily_cu = ws_daily_cu.merge(cap_daily_cu, on=["Capacity Id", "Date"], how="left")
    ws_daily_cu["cu_share"] = ws_daily_cu["ws_cu"] / ws_daily_cu["cap_total_cu"].replace(0, 1)
    ws_daily_cost = ws_daily_cu.merge(
        cost_daily_cap[["Capacity Id", "date", "daily_cost"]].rename(columns={"date": "Date"}),
        on=["Capacity Id", "Date"], how="inner"
    )
    ws_daily_cost["workspace_cost"] = ws_daily_cost["cu_share"] * ws_daily_cost["daily_cost"]
    ws_daily_cost = ws_daily_cost.merge(dim_workspaces[["Workspace Id", "Workspace Name"]].drop_duplicates(), on="Workspace Id", how="left")
    print(f"Workspace cost: {len(ws_daily_cost):,} rows over {ws_daily_cost['Date'].nunique()} overlapping days")
    if ws_daily_cost["Date"].nunique() == 0:
        print(f"  Billing: {cost_daily_cap['date'].min().date()} to {cost_daily_cap['date'].max().date()}")
        print(f"  Chargeback: {df_chargeback['Date'].min().date()} to {df_chargeback['Date'].max().date()}")
else:
    ws_daily_cost = pd.DataFrame()

---
## 3. $/Day/Workspace Ranking

In [ ]:
if not ws_daily_cost.empty:
    ws_cost_summary = (
        ws_daily_cost.groupby(["Workspace Id", "Workspace Name", "Capacity Id"])
        .agg(total_cost=pd.NamedAgg(column="workspace_cost", aggfunc="sum"),
             avg_daily_cost=pd.NamedAgg(column="workspace_cost", aggfunc="mean"),
             total_cu=pd.NamedAgg(column="ws_cu", aggfunc="sum"),
             days=pd.NamedAgg(column="Date", aggfunc="nunique"))
        .reset_index().sort_values("total_cost", ascending=False)
    )
    ws_cost_summary = ws_cost_summary.merge(dim_capacities[["Capacity Id", "Capacity Name", "SKU"]], on="Capacity Id", how="left")
    print(f"=== TOP WORKSPACES BY COST ({currency}) ===")
    print(f"Period: {ws_daily_cost['Date'].min().date()} to {ws_daily_cost['Date'].max().date()} ({ws_daily_cost['Date'].nunique()} days)\n")
    print(ws_cost_summary[["Workspace Name", "Capacity Name", "SKU", "total_cost", "avg_daily_cost", "total_cu", "days"]].head(25).to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    ta, tb = ws_cost_summary["total_cost"].sum(), cost_daily_cap["daily_cost"].sum()
    print(f"\nAllocated: {currency} {ta:,.2f} | Billed: {currency} {tb:,.2f} | Coverage: {ta/tb*100:.1f}%" if tb > 0 else "")
else:
    ws_cost_summary = pd.DataFrame()
    print("No workspace cost data.")

---
## 4. Cost per User

In [ ]:
if not ws_daily_cost.empty:
    user_ws = df_chargeback.groupby(["Capacity Id", "Workspace Id", "Date", "User"]).agg(user_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum")).reset_index()
    ws_tot = user_ws.groupby(["Capacity Id", "Workspace Id", "Date"]).agg(ws_cu=pd.NamedAgg(column="user_cu", aggfunc="sum")).reset_index()
    user_ws = user_ws.merge(ws_tot, on=["Capacity Id", "Workspace Id", "Date"], how="left")
    user_ws["share"] = user_ws["user_cu"] / user_ws["ws_cu"].replace(0, 1)
    user_cost = user_ws.merge(ws_daily_cost[["Capacity Id", "Workspace Id", "Date", "workspace_cost"]], on=["Capacity Id", "Workspace Id", "Date"], how="inner")
    user_cost["user_cost"] = user_cost["share"] * user_cost["workspace_cost"]
    user_summary = user_cost.groupby("User").agg(
        total_cost=pd.NamedAgg(column="user_cost", aggfunc="sum"),
        total_cu=pd.NamedAgg(column="user_cu", aggfunc="sum"),
        workspaces=pd.NamedAgg(column="Workspace Id", aggfunc="nunique"),
        days=pd.NamedAgg(column="Date", aggfunc="nunique")
    ).reset_index().sort_values("total_cost", ascending=False)
    user_summary["avg_daily"] = user_summary["total_cost"] / user_summary["days"].replace(0, 1)
    print(f"=== TOP USERS BY COST ({currency}) ===")
    print(user_summary.head(20).to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
else:
    print("No cost data.")

---
## 5. Cost per Domain

In [ ]:
if not ws_daily_cost.empty and "Domain" in df_chargeback.columns and df_chargeback["Domain"].notna().any():
    dom_ws = df_chargeback[df_chargeback["Domain"].notna()].groupby(["Capacity Id", "Workspace Id", "Date", "Domain", "Subdomain"]).agg(dom_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum")).reset_index()
    ws_tot = dom_ws.groupby(["Capacity Id", "Workspace Id", "Date"]).agg(ws_cu=pd.NamedAgg(column="dom_cu", aggfunc="sum")).reset_index()
    dom_ws = dom_ws.merge(ws_tot, on=["Capacity Id", "Workspace Id", "Date"], how="left")
    dom_ws["share"] = dom_ws["dom_cu"] / dom_ws["ws_cu"].replace(0, 1)
    dom_cost = dom_ws.merge(ws_daily_cost[["Capacity Id", "Workspace Id", "Date", "workspace_cost"]], on=["Capacity Id", "Workspace Id", "Date"], how="inner")
    dom_cost["domain_cost"] = dom_cost["share"] * dom_cost["workspace_cost"]
    dom_summary = dom_cost.groupby(["Domain", "Subdomain"]).agg(
        total_cost=pd.NamedAgg(column="domain_cost", aggfunc="sum"),
        total_cu=pd.NamedAgg(column="dom_cu", aggfunc="sum"),
        workspaces=pd.NamedAgg(column="Workspace Id", aggfunc="nunique")
    ).reset_index().sort_values("total_cost", ascending=False)
    print(f"=== DOMAIN COST ({currency}) ===")
    print(dom_summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
else:
    print("No domain data or no cost data.")

---
## 6. Cost Efficiency ($/CU-second)

In [ ]:
if not cost_daily_cap.empty:
    cap_cu = df_chargeback.groupby(["Capacity Id", "Date"]).agg(total_cu=pd.NamedAgg(column="CU (s)", aggfunc="sum")).reset_index()
    eff = cost_daily_cap.rename(columns={"date": "Date"}).merge(cap_cu, on=["Capacity Id", "Date"], how="inner")
    if not eff.empty:
        eff_s = eff.groupby(["Capacity Name", "SKU"]).agg(cost=pd.NamedAgg(column="daily_cost", aggfunc="sum"), cu=pd.NamedAgg(column="total_cu", aggfunc="sum"), days=pd.NamedAgg(column="Date", aggfunc="nunique")).reset_index()
        eff_s["per_cu_s"] = eff_s["cost"] / eff_s["cu"].replace(0, 1)
        print(f"=== COST EFFICIENCY ({currency}/CU-second) ===")
        for _, r in eff_s.iterrows():
            print(f"  {r['Capacity Name']} ({r['SKU']}): {currency} {r['per_cu_s']:.8f}/CU-s = {currency} {r['per_cu_s']*3600:.4f}/CU-hr ({r['days']}d)")
    else:
        print("No date overlap.")
else:
    print("No billing data.")

---
## 7. Monthly Cost Trend

In [ ]:
if not df_billing.empty:
    df_billing["_month"] = df_billing["date"].dt.to_period("M")
    mc = df_billing.groupby(["capacity_name", "_month"]).agg(cost=pd.NamedAgg(column="costInBillingCurrency", aggfunc="sum"), days=pd.NamedAgg(column="date", aggfunc="nunique")).reset_index().sort_values(["capacity_name", "_month"])
    mc["prev"] = mc.groupby("capacity_name")["cost"].shift(1)
    mc["pct"] = ((mc["cost"] - mc["prev"]) / mc["prev"].replace(0, 1) * 100).round(1)
    print(f"=== MONTHLY COST TREND ({currency}) ===")
    for cap in mc["capacity_name"].unique():
        print(f"\n  {cap}:")
        for _, r in mc[mc["capacity_name"] == cap].iterrows():
            c = f" ({r['pct']:+.1f}%)" if pd.notna(r["pct"]) and pd.notna(r["prev"]) else ""
            p = " (partial)" if r["days"] < 28 else ""
            print(f"    {r['_month']}: {currency} {r['cost']:>10,.2f}  ({r['days']}d){p}{c}")
else:
    print("No billing data.")

---
## 8. Reserved vs On-Demand

In [ ]:
if not df_billing.empty and "pricingModel" in df_billing.columns:
    pp = df_billing.groupby(["capacity_name", "pricingModel"])["costInBillingCurrency"].sum().reset_index()
    pp = pp.pivot_table(index="capacity_name", columns="pricingModel", values="costInBillingCurrency", fill_value=0, aggfunc="sum")
    pp["Total"] = pp.sum(axis=1)
    for col in [c for c in pp.columns if c != "Total"]:
        pp[f"{col} %"] = (pp[col] / pp["Total"].replace(0, 1) * 100).round(1)
    print(f"=== PRICING MODEL ({currency}) ===")
    print(pp.to_string(float_format=lambda x: f"{x:,.2f}"))
else:
    print("No pricing model data.")

---
## 9. Meter Group Breakdown (Compute vs Storage vs AI)

In [ ]:
if not df_billing.empty and "meter_group" in df_billing.columns:
    mg = df_billing.groupby(["capacity_name", "meter_group"])["costInBillingCurrency"].sum().reset_index()
    mg = mg.pivot_table(index="capacity_name", columns="meter_group", values="costInBillingCurrency", fill_value=0, aggfunc="sum")
    mg["Total"] = mg.sum(axis=1)
    for col in [c for c in mg.columns if c != "Total"]:
        mg[f"{col} %"] = (mg[col] / mg["Total"].replace(0, 1) * 100).round(1)
    print(f"=== COST BY METER GROUP ({currency}) ===")
    print(mg.to_string(float_format=lambda x: f"{x:,.2f}"))
else:
    print("No meter_group column. Re-run Data Prep.")

---
## 10. Executive Summary

In [ ]:
for cap_id in dim_capacities["Capacity Id"].unique():
    cr = dim_capacities[dim_capacities["Capacity Id"] == cap_id].iloc[0]
    cn, sku, cores = cr.get("Capacity Name", cap_id[:12]), cr.get("SKU", "?"), cr.get("Core count", "?")
    if not cost_daily_cap.empty:
        cc = cost_daily_cap[cost_daily_cap["Capacity Id"] == cap_id]
        tc, ad, cd = cc["daily_cost"].sum(), cc["daily_cost"].mean() if not cc.empty else 0, cc["date"].nunique() if "date" in cc.columns else 0
    else:
        tc = ad = cd = 0
    cb = df_chargeback[df_chargeback["Capacity Id"] == cap_id]
    cu, users, wss = cb["CU (s)"].sum(), cb["User"].nunique(), cb["Workspace Id"].nunique()
    top = pd.DataFrame()
    if "ws_cost_summary" in dir() and not ws_cost_summary.empty:
        top = ws_cost_summary[ws_cost_summary["Capacity Id"] == cap_id].nlargest(5, "total_cost")
    print(f"\n{'='*70}")
    print(f"CAPACITY: {cn} ({sku}, {cores} cores)")
    print(f"{'='*70}")
    if tc > 0:
        print(f"  COST ({cd}d): {currency} {tc:,.2f} total | {currency} {ad:,.2f}/day | {currency} {ad*30:,.2f}/mo")
    print(f"  USAGE: {cu:,.0f} CU-s | {users} users | {wss} workspaces")
    if tc > 0 and cu > 0:
        cps = tc / cu
        print(f"  EFFICIENCY: {currency} {cps:.8f}/CU-s = {currency} {cps*3600:.4f}/CU-hr")
    if not top.empty:
        print(f"  TOP WORKSPACES:")
        for _, ws in top.iterrows():
            print(f"    {ws['Workspace Name']}: {currency} {ws['total_cost']:,.2f} ({currency} {ws['avg_daily_cost']:.2f}/day)")